# 第一级：规则清洗

In [10]:
import os
import re
import json
from langdetect import detect

save_dir = "../../saved/train_expansion"
files = os.listdir(save_dir)
tool_calls_stat = {}
droped_count = 0
total_count = 0
train_data = []

stat_turns = []
for filename in files:
    if filename.endswith(".json"):
        info = json.load(open(os.path.join(save_dir, filename)))
        question = info["question"]
        msg = info["messages"][-1]["content"]
        stats = info["stats"]
        turn = stats.pop("turns", 0) # 对话轮次
        num_calls = sum(stats.values()) # 工具调用的总次数
        
        if turn > 13 or num_calls == 0: # 调用超过13次或者没有调用工具
            droped_count += 1
            continue
            
        elif "exceeds the limit" in msg or "reached the maximum" in msg: # 超过最大对话轮次/上下文最大长度
            droped_count += 1
            continue
            
        elif  "没有找到" in msg:   # 答案里面"没有找到"（query不合理或者query缺少信息）
            droped_count += 1  
            continue
            
        elif  detect(msg) == "en":  # 答案语言是英文
            droped_count += 1
            continue
            
        elif len(msg) < 100:     # 答案太简短
            droped_count += 1
            continue

        # 记录工具调用情况
        for name, value in stats.items():
            if name not in tool_calls_stat:
                tool_calls_stat[name] = 0
            tool_calls_stat[name] += 1

        new_info = {
            "qid": filename.split(".")[0],
            "question": info["question"],
            "answer": json.dumps(info["messages"], ensure_ascii=False)
        }

        train_data.append(new_info)
        stat_turns.append(turn)
            
        total_count += 1

# 统计工具的调用次数
for k, v in tool_calls_stat.items():
    print(k, v)
print()
print(f"total count: {total_count}, dropped count: {droped_count}")

train_tickets_search 292
search 1165
visit 558
poi_search 895
around_search 350
route_planning 601
weather_search 147
flights_search 217

total count: 1698, dropped count: 133


In [11]:
import numpy as np

for k in [50, 60, 70, 80, 90, 95]:
    print(f"{k}分位点: {np.percentile(stat_turns, k)}")

50分位点: 5.0
60分位点: 5.0
70分位点: 6.0
80分位点: 7.0
90分位点: 9.0
95分位点: 10.0


In [12]:
import os
import re
import json
from langdetect import detect

save_dir = "../../saved/test"
files = os.listdir(save_dir)
tool_calls_stat = {}
droped_count = 0
total_count = 0
test_data = []
for filename in files:
    if filename.endswith(".json"):
        info = json.load(open(os.path.join(save_dir, filename)))
        question = info["question"]
        msg = info["messages"][-1]["content"]
        stats = info["stats"]
        turn = stats.pop("turns", 0)
        num_calls = sum(stats.values())
        if turn > 13 or num_calls == 0: # 调用超过13次或者没有调用工具
            droped_count += 1
            continue
        elif "exceeds the limit" in msg or "reached the maximum" in msg: # 超过最大对话轮次
            droped_count += 1
            continue
        elif  "没有找到" in msg:   # 答案里面"没有找到"（query不合理或者query缺少信息）
            droped_count += 1  
            continue
        elif  detect(msg) == "en":   # 答案语言全是英文
            droped_count += 1
            continue
        elif len(msg) < 100:     # 答案太简短
            droped_count += 1
            continue

        # 记录工具调用情况
        for name, value in stats.items():
            if name not in tool_calls_stat:
                tool_calls_stat[name] = 0
            tool_calls_stat[name] += 1

        new_info = {
            "qid": filename.split(".")[0],
            "question": info["question"],
            "answer": json.dumps(info["messages"], ensure_ascii=False)
        }
            
        total_count += 1
        test_data.append(new_info)

# 统计工具的调用次数
for k, v in tool_calls_stat.items():
    print(k, v)
print()
print(f"total count: {total_count}, dropped count: {droped_count}")

poi_search 62
around_search 28
route_planning 48
train_tickets_search 27
flights_search 12
search 45
visit 11
weather_search 10

total count: 86, dropped count: 14


In [13]:
train_data[0]

{'qid': '0048a90ec0995c56f5439b1e4cdd810f',
 'question': '济南出发，想去泰山看日出，坐火车怎么安排时间？',
 'answer': '[{"role": "system", "content": "您是一名有用的旅行规划助手。您的核心职责是对用户的旅行或者游玩攻略进行规划和回答。对于每个请求，您需要综合来自可靠、多样化来源的信息，以提供全面、准确和实时的答复。请提前规划需要调用哪些工具，以便完整获取所需要的信息。当您收集到足够的信息并准备好提供答复时，请返回给用户完整的答案，并将答案放在 `<answer>Your Complete Answer</answer>` 内。\\n\\n当前日期：2026-04-04\\n最大可调用50轮工具"}, {"role": "user", "content": "济南出发，想去泰山看日出，坐火车怎么安排时间？"}, {"content": "您想去泰山看日出，这是一个很棒的旅行计划！让我先帮您查询济南到泰安的火车时刻表以及泰山日出的相关信息。\\n", "refusal": null, "role": "assistant", "annotations": null, "audio": null, "function_call": null, "tool_calls": [{"id": "call_function_sxa0ghm1f7z9_1", "function": {"arguments": "{\\"date\\": \\"2026-04-04\\", \\"from_city\\": \\"济南\\", \\"to_city\\": \\"泰安\\"}", "name": "train_tickets_search"}, "type": "function", "index": 0}, {"id": "call_function_sxa0ghm1f7z9_2", "function": {"arguments": "{\\"query\\": [\\"泰山日出时间 最佳观赏时间\\", \\"济南到泰安火车站时刻表\\", \\"泰安站到泰山景区交通方式\\"]}", "name": "search"}, "type": "function", "index

## 格式化messages并保存

In [14]:
save_dir = "../../data/clean/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

iterators = [
    [train_data, open(save_dir + "train.jsonl", "w")],
    [test_data, open(save_dir + "test.jsonl", "w")],
]
all_data = []
for data, handler in iterators:
    for item in data:
        messages = json.loads(item["answer"])
        conversations = []
        for message in messages:
            if message["role"] == "system":
                message["content"] = message["content"].replace("最大可调用50轮工具", "最大可调用13轮工具")
                info = {
                    "role": "system",
                    "content": message["content"]
                }
            elif message["role"] == "user":
                info = {
                    "role": "user",
                    "content": message["content"]
                }
            elif message["role"] == "assistant":
                # 处理tool calls和answer
                reasoning_body = message["reasoning_details"]
                reasoning_content = "\n".join([reasoning_body[t]["text"] for t in range(len(reasoning_body))])
                if reasoning_content.strip():
                    reasoning_content = f"<think>\n{reasoning_content}\n</think>\n\n"
                else:
                    reasoning_content = ""
                if message["tool_calls"]:
                    content = json.dumps(message["tool_calls"], ensure_ascii=False)
                    info = {
                        "role": "assistant", 
                        "content": f"{reasoning_content}<tool_call>\n{content}\n</tool_call>"
                    }
                elif message["content"].strip():
                    # 处理答案
                    content = message["content"]
                    info = {
                        "role": "assistant", 
                        "content": f"{reasoning_content}{content}"
                    }   
                
            elif message["role"] == "tool":
                # 处理工具返回
                info = {
                    "role": "user",
                    "content": f"<tool_response>\n{message['content']}\n</tool_response>"
                }
            conversations.append(info)
        body = {
            "id": item["qid"],
            "conversations": conversations
        }
        all_data.append(body)
        handler.write(json.dumps(body, ensure_ascii=False) + "\n")

In [18]:
import random
random.choice(all_data)["conversations"]

[{'role': 'system',
  'content': '您是一名有用的旅行规划助手。您的核心职责是对用户的旅行或者游玩攻略进行规划和回答。对于每个请求，您需要综合来自可靠、多样化来源的信息，以提供全面、准确和实时的答复。请提前规划需要调用哪些工具，以便完整获取所需要的信息。当您收集到足够的信息并准备好提供答复时，请返回给用户完整的答案，并将答案放在 `<answer>Your Complete Answer</answer>` 内。\n\n当前日期：2026-04-03\n最大可调用13轮工具'},
 {'role': 'user', 'content': '澳门到南京明天坐飞机和高铁大概多少钱'},
 {'role': 'assistant',
  'content': '<think>\n用户询问从澳门到南京明天（2026年4月4日）乘坐飞机和高铁的价格。\n\n我需要分别查询：\n1. 航班信息：从澳门到南京的航班（可能需要从珠海或澳门机场出发到南京）\n2. 高铁信息：从澳门到南京的高铁（可能需要从珠海或广州南出发）\n\n让我先查询航班和火车票信息。不过我需要注意，澳门到南京可能没有直达的高铁，可能需要中转。我先查询航班和火车票看看情况。\n\n让我同时查询航班和火车票信息。\n</think>\n\n<tool_call>\n[{"id": "call_function_f4m9o9f7vvw6_1", "function": {"arguments": "{\\"date\\": \\"2026-04-04\\", \\"from_city\\": \\"澳门\\", \\"to_city\\": \\"南京\\"}", "name": "flights_search"}, "type": "function", "index": 0}, {"id": "call_function_f4m9o9f7vvw6_2", "function": {"arguments": "{\\"date\\": \\"2026-04-04\\", \\"from_city\\": \\"珠海\\", \\"to_city\\": \\"南京\\"}", "name": "train_tickets_search"}, "type": "function", 